# Module 31 — CTR Prediction with Logistic Regression

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Scikit-learn, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 13 (Logistic Regression)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Ad Impression Data](#3-synthetic-ad-impression-data)
4. [Feature Engineering](#4-feature-engineering)
5. [Logistic Regression Model](#5-logistic-regression-model)
6. [Calibration](#6-calibration)
7. [Evaluation](#7-evaluation)
8. [Visualization](#8-visualization)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why CTR prediction for ads?

Click-Through Rate (CTR) prediction is **critical** for ad platforms:
- **Revenue optimization**: Show ads users are likely to click.
- **User experience**: Relevant ads improve engagement.
- **Advertiser ROI**: Better targeting means higher conversions.

**Applications at Yandex**:
1. **Search ads**: Predict CTR for search result ads.
2. **Display ads**: Predict CTR for banner ads.
3. **Native ads**: Predict CTR for sponsored content.

**Key metrics**:
- **LogLoss**: Primary metric for CTR calibration.
- **AUC-ROC**: Discrimination ability.
- **Calibration**: Predicted probabilities match actual CTR.

In this notebook we will:
1. Generate synthetic ad impression data.
2. Engineer features for CTR prediction.
3. Train logistic regression with feature hashing.
4. Calibrate probabilities and evaluate.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    classification_report
)
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction import FeatureHasher

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Ad Impression Data

In [ ]:
# Generate synthetic ad impression data
N_IMPRESSIONS = 500000
CTR_BASE = 0.02  # 2% base CTR

# Generate features
data = {
    'user_id': np.random.randint(0, 100000, N_IMPRESSIONS),
    'ad_id': np.random.randint(0, 10000, N_IMPRESSIONS),
    'publisher_id': np.random.randint(0, 1000, N_IMPRESSIONS),
    'ad_position': np.random.choice([1, 2, 3, 4, 5], N_IMPRESSIONS, p=[0.3, 0.25, 0.2, 0.15, 0.1]),
    'hour': np.random.randint(0, 24, N_IMPRESSIONS),
    'day_of_week': np.random.randint(0, 7, N_IMPRESSIONS),
    'device': np.random.choice(['mobile', 'desktop', 'tablet'], N_IMPRESSIONS, p=[0.6, 0.3, 0.1]),
    'ad_category': np.random.choice(['electronics', 'clothing', 'food', 'travel', 'finance'], N_IMPRESSIONS),
    'user_age': np.random.normal(35, 10, N_IMPRESSIONS).clip(18, 70).astype(int),
    'user_gender': np.random.choice(['M', 'F'], N_IMPRESSIONS),
}

df = pd.DataFrame(data)

# Generate CTR labels (correlated with features)
ctr_prob = (
    CTR_BASE
    + 0.01 * (df['ad_position'] == 1).astype(float)  # Top position
    + 0.005 * (df['device'] == 'mobile').astype(float)  # Mobile higher CTR
    - 0.003 * (df['hour'] > 22).astype(float)  # Late night lower CTR
    + np.random.normal(0, 0.005, N_IMPRESSIONS)
)
ctr_prob = ctr_prob.clip(0.001, 0.999)

df['clicked'] = np.random.binomial(1, ctr_prob)

print(f"✓ Generated {N_IMPRESSIONS:,} ad impressions")
print(f"  CTR: {df['clicked'].mean()*100:.2f}%")
print(f"  Users: {df['user_id'].nunique():,}")
print(f"  Ads: {df['ad_id'].nunique():,}")
print(f"  Publishers: {df['publisher_id'].nunique():,}")

---
## §4 · Feature Engineering

In [ ]:
# Feature engineering for CTR
# Encode categorical features
df_encoded = pd.get_dummies(df, columns=['device', 'ad_category', 'user_gender'], drop_first=True)

# User historical CTR
user_ctr = df.groupby('user_id')['clicked'].mean().rename('user_hist_ctr')
df_encoded = df_encoded.merge(user_ctr, on='user_id', how='left')

# Ad historical CTR
ad_ctr = df.groupby('ad_id')['clicked'].mean().rename('ad_hist_ctr')
df_encoded = df_encoded.merge(ad_ctr, on='ad_id', how='left')

# Publisher historical CTR
pub_ctr = df.groupby('publisher_id')['clicked'].mean().rename('pub_hist_ctr')
df_encoded = df_encoded.merge(pub_ctr, on='publisher_id', how='left')

# Time features
df_encoded['is_weekend'] = (df_encoded['day_of_week'] >= 5).astype(int)
df_encoded['is_night'] = ((df_encoded['hour'] >= 22) | (df_encoded['hour'] <= 6)).astype(int)

feature_cols = [c for c in df_encoded.columns if c not in ['clicked', 'user_id', 'ad_id', 'publisher_id']]

print(f"✓ Engineered {len(feature_cols)} features")
print(f"\nFeature examples:")
for feat in feature_cols[:10]:
    print(f"  - {feat}")

---
## §5 · Logistic Regression Model

In [ ]:
# Prepare data
X = df_encoded[feature_cols]
y = df_encoded['clicked']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression (using SGD for scalability)
model = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=1e-4,
    max_iter=100,
    random_state=42
)

print("Training CTR model...")
model.fit(X_train_scaled, y_train)

print(f"✓ Model trained")
print(f"  Features: {len(feature_cols)}")
print(f"  Training samples: {len(y_train):,}")

---
## §6 · Calibration

In [ ]:
# Calibrate probabilities
calibrated_model = CalibratedClassifierCV(model, method='sigmoid', cv=3)
calibrated_model.fit(X_train_scaled, y_train)

y_prob = calibrated_model.predict_proba(X_test_scaled)[:, 1]

# Compute calibration metrics
brier = brier_score_loss(y_test, y_prob)
logloss = log_loss(y_test, y_prob)

print(f"✓ Calibration complete")
print(f"  Brier Score: {brier:.4f}")
print(f"  LogLoss: {logloss:.4f}")
print(f"\n💡 Lower LogLoss indicates better calibration")

---
## §7 · Evaluation

In [ ]:
# Evaluate model
auc = roc_auc_score(y_test, y_prob)

print("=" * 70)
print("CTR MODEL EVALUATION")
print("=" * 70)
print(f"\nROC-AUC: {auc:.4f}")
print(f"LogLoss: {logloss:.4f}")
print(f"Brier Score: {brier:.4f}")
print(f"\nPredicted CTR: {y_prob.mean()*100:.2f}%")
print(f"Actual CTR: {y_test.mean()*100:.2f}%")

---
## §8 · Visualization

In [ ]:
# Visualize CTR prediction
fig = make_subplots(rows=1, cols=2, subplot_titles=['Predicted vs Actual CTR', 'CTR by Position'])

# Predicted vs Actual CTR (binned)
n_bins = 20
df_eval = pd.DataFrame({'actual': y_test, 'predicted': y_prob})
df_eval['bin'] = pd.cut(df_eval['predicted'], bins=n_bins)
bin_stats = df_eval.groupby('bin', observed=True).agg(
    predicted_mean='predicted',
    actual_mean='actual',
    count='actual'
).apply(lambda x: x.mean())

fig.add_trace(go.Scatter(
    x=bin_stats['predicted_mean'],
    y=bin_stats['actual_mean'],
    mode='markers',
    name='Calibration',
    marker=dict(size=10)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[0, 0.1], y=[0, 0.1],
    mode='lines',
    name='Perfect calibration',
    line=dict(dash='dash', color='red')
), row=1, col=1)

# CTR by position
position_ctr = df.groupby('ad_position')['clicked'].mean()
fig.add_trace(go.Bar(
    x=position_ctr.index,
    y=position_ctr.values * 100,
    name='CTR by Position'
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Model calibration is important for accurate CTR prediction")
print("   - Top ad positions have higher CTR")
print("   - Logistic regression provides good baseline for CTR")

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for Ad Platform Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Ad ranking, bid optimization, budget allocation |
> | **Model** | Logistic regression (baseline), GBDT (production) |
> | **Features** | User history, ad features, context (time, device) |
> | **Calibration** | Critical for accurate bid calculations |
> | **Evaluation** | LogLoss (primary), AUC-ROC, calibration plots |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Ad ranking formula**:
>    ```
>    Rank Score = CTR × Bid × Quality Score
>    ```
>
> 2. **Feature categories**:
>    | Category | Examples |
>    |----------|--------|
>    | User | Age, gender, history, device |
>    | Ad | Category, creative, landing page |
>    | Context | Time, position, publisher |
>
> 3. **Production tips**:
>    - Use online learning for real-time updates
>    - Calibrate probabilities daily
>    - A/B test model changes against revenue metrics
>
> ### 🔑 Key Takeaways
>
> 1. **CTR prediction is revenue-critical** for ad platforms.
> 2. **Calibration is essential** for accurate bid calculations.
> 3. **Logistic regression** provides a strong baseline.
> 4. **Feature engineering** (user/ad history) is key.
> 5. **LogLoss is the primary metric** for CTR models.